In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from scipy.interpolate import interp1d
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold # NEW: Swapped to K-Fold
import copy
import random
import extinction
import joblib

 IMPORTS FOR SLIDE GENERATION

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

==========================================<br>
 HARDWARE CHECK<br>
==========================================

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("\n" + "="*60)
if device.type == 'cuda':
    print(f" GPU DETECTED: Utilizing {torch.cuda.get_device_name(0)}")
else:
    print("  NO GPU DETECTED: Falling back to CPU.")
print("="*60 + "\n")

==========================================<br>
CONFIGURATION (v11 - Anti-Overfitting)<br>
==========================================

In [ ]:
CONFIG = {
    'DATA_DIR': 'DATA/', 
    'EPOCHS': 75, # Slightly reduced per-fold to save time
    'BATCH_SIZE': 32, 
    'MAX_SEQ_LENGTH': 260,    # Forces all time-series to have exactly 200 steps
    'DAYS_BEFORE_PEAK': 30.0, # Time window start
    'DAYS_AFTER_PEAK': 100.0, # Time window end
    'FLUX_SCALE_FACTOR': 1500.0, # Normalizes raw brightness values to ~[0,1] range
    'NUM_CHANNELS': 24,       # 6 filters x 4 features (flux, diff, mask, error)
    'DROPOUT_RATE': 0.5,      # Increased to 50% to fight overfitting
    'FOCAL_GAMMA': 1.5,       # Focuses loss on hard-to-predict examples
    'FOCAL_ALPHA': 0.5,       # Balances the baseline class weight
    'EARLY_STOP_PATIENCE': 10,
    'NUM_STATIC_FEATURES': 14,
    'K_FOLDS': 5              # NEW: 5-Fold Cross Validation
}

SVO Filter Profile Service Effective Wavelengths (Used for Dust/Reddening Correction)

In [ ]:
EFF_WL = {
    'u': np.array([3641.0]), 'g': np.array([4704.0]), 'r': np.array([6155.0]),
    'i': np.array([7504.0]), 'z': np.array([8695.0]), 'y': np.array([10056.0])
}

Determinism for reproducibility

In [ ]:
random.seed(42); np.random.seed(42); torch.manual_seed(42)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(42)

==========================================<br>
FOCAL LOSS: Heavily penalizes missing TDEs<br>
==========================================

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=CONFIG['FOCAL_ALPHA'], gamma=CONFIG['FOCAL_GAMMA']):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean()

==========================================<br>
1. MODEL: Dual-Branch CNN + MLP (Shrunk for small data)<br>
==========================================

In [ ]:
class MultiModalCNN(nn.Module):
    def __init__(self, num_static_features=CONFIG['NUM_STATIC_FEATURES'], num_bands=CONFIG['NUM_CHANNELS']):
        super().__init__()
        
        # SEQUENCE BRAIN (CNN): Looks for flares, cooling trends, and temporal patterns
        # NEW: Drastically reduced channel counts to [16, 32, 64] and added 1D Dropout
        self.cnn = nn.Sequential(
            nn.Conv1d(num_bands, 16, kernel_size=7, padding=3),
            nn.BatchNorm1d(16), nn.ReLU(),
            nn.Dropout1d(0.2), # Drops entire channels randomly to prevent memorization
            
            nn.Conv1d(16, 32, kernel_size=5, padding=2),
            nn.BatchNorm1d(32), nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout1d(0.2),
            
            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1) # Squashes the time dimension down to a single summary vector
        )
        
        # STATIC BRAIN (MLP): Processes global physics data (Redshift, Colors, Duration)
        self.mlp = nn.Sequential(
            nn.Linear(num_static_features, 64), nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.3),
            nn.Linear(64, 128), nn.ReLU()  
        )
        
        # CLASSIFIER: Fuses the output of both brains to make a final 0 or 1 decision
        self.classifier = nn.Sequential(
            nn.Linear(64 + 128 + num_static_features, 128), nn.ReLU(),
            nn.Dropout(CONFIG['DROPOUT_RATE']),
            nn.Linear(128, 2)
        )
    def forward(self, x_seq, x_static):
        # 1. Process Sequences
        cnn_out = torch.flatten(self.cnn(torch.nan_to_num(x_seq)), 1)
        
        # 2. Process Statics
        x_static_clean = torch.nan_to_num(x_static)
        mlp_out = self.mlp(x_static_clean)
        
        # 3. Skip Connection: Passes raw redshift/physics data directly to the end
        combined = torch.cat((cnn_out, mlp_out, x_static_clean), dim=1)
        
        return self.classifier(combined)

==========================================<br>
2. PRE-PROCESSING: Physics corrections & caching<br>
==========================================

In [ ]:
def preprocess_and_cache(log_path, lightcurves_dir, prefix="train", is_train=True):
    cache_file = os.path.join(CONFIG['DATA_DIR'], f"{prefix}_cache_v11.pt")
    scaler_file = os.path.join(CONFIG['DATA_DIR'], 'static_scaler_v11.pkl')
    
    # Check if we already did the hard math
    if os.path.exists(cache_file) and (not is_train or os.path.exists(scaler_file)):
        print(f"Loading cache for {prefix}...")
        data = torch.load(cache_file)
        if is_train:
            scaler = joblib.load(scaler_file)
            return data, scaler
        return data, None
    
    print(f"Building cache...")
    log_df = pd.read_csv(log_path)
    unique_splits = log_df['split'].dropna().unique()
    df_list = [pd.read_csv(os.path.join(lightcurves_dir, s, f"{prefix}_full_lightcurves.csv")) 
               for s in unique_splits if os.path.exists(os.path.join(lightcurves_dir, s, f"{prefix}_full_lightcurves.csv"))]
    lc_df = pd.concat(df_list, ignore_index=True).fillna(0.0)
    grouped_lc = lc_df.groupby('object_id')
    filters = ['u', 'g', 'r', 'i', 'z', 'y']
    all_seqs, all_statics, all_targets, all_ids = [], [], [], []
    for idx, row in log_df.iterrows():
        obj_id = row['object_id']
        ebv = row['EBV'] # Milky way dust
        z_val = row['Z'] if not pd.isna(row['Z']) else 0.0
        z_err = row['Z_err'] if not pd.isna(row['Z_err']) else 0.0
        
        # Redshift Augmentation: Smears the redshift slightly during training
        if is_train and z_err == 0.0:
            z_err = np.random.uniform(0.01, 0.08)
            z_val = max(0.0, z_val + np.random.normal(0, z_err))
        
        # Initialize default values
        peak_flux_norm = n_obs_norm = duration_norm = blue_red_ratio = 0.0
        u_minus_g = g_minus_r = log_peak_flux_norm = obs_density = 0.0
        rise_days_norm = decay_days_norm = asymmetry = flux_std_norm = active_bands = 0.0
        seq_tensor = np.zeros((CONFIG['NUM_CHANNELS'], CONFIG['MAX_SEQ_LENGTH']), dtype=np.float32)
        if obj_id in grouped_lc.groups:
            obj_lc = grouped_lc.get_group(obj_id).sort_values('Time (MJD)')
            if not obj_lc.empty:
                t_max_idx = obj_lc['Flux'].idxmax()
                t_max = obj_lc.loc[t_max_idx, 'Time (MJD)']
                t_first, t_last = obj_lc['Time (MJD)'].min(), obj_lc['Time (MJD)'].max()
                
                dered_max_per_band, g_mean_dered, r_mean_dered = [], 0.0, 0.0
                u_max = g_max = r_max = 0.0
                n_obs_total = len(obj_lc)
                duration_days = t_last - t_first if n_obs_total > 1 else 0.0
                
                # Dust Dereddening (Physics Correction)
                for f in filters:
                    band_data = obj_lc[obj_lc['Filter'] == f]
                    if len(band_data) > 0:
                        # Fitzpatrick99: Removes Milky Way dust dimming using the filter's wavelength
                        A_lambda = extinction.fitzpatrick99(EFF_WL[f], ebv * 3.1)[0]
                        f_corr = band_data['Flux'].values * (10 ** (A_lambda / 2.5))
                        dered_max_per_band.append(f_corr.max())
                        if f == 'u': u_max = f_corr.max()
                        if f == 'g': g_max, g_mean_dered = f_corr.max(), f_corr.mean()
                        if f == 'r': r_max, r_mean_dered = f_corr.max(), f_corr.mean()
                # Feature Engineering
                peak_flux_global = max(dered_max_per_band) if dered_max_per_band else 0.0
                peak_flux_norm = peak_flux_global / CONFIG['FLUX_SCALE_FACTOR']
                blue_red_ratio = g_mean_dered / (r_mean_dered + 1e-8) if r_mean_dered > 0 else 1.0
                u_minus_g = u_max - g_max
                g_minus_r = g_max - r_max
                obs_density = n_obs_total / (duration_days + 1.0)
                
                rise_days, decay_days = t_max - t_first, t_last - t_max
                asymmetry = rise_days / (rise_days + decay_days + 1e-8)
                active_bands = sum(1 for f in filters if len(obj_lc[obj_lc['Filter'] == f]) > 2)
                
                grid = np.linspace(t_max - CONFIG['DAYS_BEFORE_PEAK'], t_max + CONFIG['DAYS_AFTER_PEAK'], CONFIG['MAX_SEQ_LENGTH'])
                # Grid Interpolation (Converting scattered points to strict matrices)
                for i, f in enumerate(filters):
                    band_data = obj_lc[obj_lc['Filter'] == f]
                    if len(band_data) > 2:
                        A_lambda = extinction.fitzpatrick99(EFF_WL[f], ebv * 3.1)[0]
                        f_corr = band_data['Flux'].values * (10 ** (A_lambda / 2.5))
                        peak_flux = f_corr.max() if len(f_corr) > 0 else 1.0
                        
                        interp_flux = interp1d(band_data['Time (MJD)'].values, f_corr, kind='linear', bounds_error=False, fill_value=0.0)
                        flux_norm = interp_flux(grid) / (peak_flux if peak_flux > 0 else 1.0)
                        
                        interp_err = interp1d(band_data['Time (MJD)'].values, band_data['Flux_err'].values, kind='linear', bounds_error=False, fill_value=0.0)
                        
                        seq_tensor[i, :] = flux_norm                # Channel 0-5: Dereddened, normalized flux
                        seq_tensor[i + 6, :] = np.diff(flux_norm, prepend=flux_norm[0]) # Channel 6-11: First derivative (slope)
                        mask_interp = interp1d(band_data['Time (MJD)'].values, np.ones(len(band_data)), kind='nearest', bounds_error=False, fill_value=0.0)
                        seq_tensor[i + 12, :] = mask_interp(grid)   # Channel 12-17: Observation masks (1 if observed, 0 if interpolated)
                        seq_tensor[i + 18, :] = interp_err(grid) / (peak_flux if peak_flux > 0 else 1.0) # Channel 18-23: Errors
        static_arr = np.array([
            z_val, z_err, peak_flux_norm, min(n_obs_total/300, 1.0), min(duration_days/300, 1.0), 
            blue_red_ratio, u_minus_g, g_minus_r, np.log1p(peak_flux_norm), obs_density,
            min(rise_days/300, 1.0), min(decay_days/300, 1.0), asymmetry, 
            np.std(obj_lc['Flux'].values)/(peak_flux_global+1e-8) if n_obs_total>1 else 0.0, active_bands
        ], dtype=np.float32)[:CONFIG['NUM_STATIC_FEATURES']]
        
        all_seqs.append(seq_tensor)
        all_statics.append(static_arr)
        all_targets.append(row['target'] if is_train else 0)
        all_ids.append(obj_id)
    print("Cache complete!")
    cached_data = {
        'ids': all_ids,
        'seqs': torch.tensor(np.array(all_seqs), dtype=torch.float32),
        'statics': torch.tensor(np.array(all_statics), dtype=torch.float32),
        'targets': torch.tensor(np.array(all_targets), dtype=torch.long)
    }
    torch.save(cached_data, cache_file)
    
    if is_train:
        scaler = StandardScaler()
        scaler.fit(cached_data['statics'].numpy())
        joblib.dump(scaler, scaler_file)
        return cached_data, scaler
    return cached_data, None

==========================================<br>
3. DATASET & HEAVY AUGMENTATION<br>
==========================================

In [ ]:
class FastAstroDataset(Dataset):
    def __init__(self, data_dict, indices=None, is_train=True):
        self.is_train = is_train
        if indices is not None:
            self.seqs, self.statics = data_dict['seqs'][indices], data_dict['statics'][indices]
            self.targets, self.ids = data_dict['targets'][indices], [data_dict['ids'][i] for i in indices]
        else:
            self.seqs, self.statics = data_dict['seqs'], data_dict['statics']
            self.targets, self.ids = data_dict['targets'], data_dict['ids']
    def __len__(self): return len(self.targets)
    def __getitem__(self, idx):
        seq = self.seqs[idx].clone()
        stat = self.statics[idx].clone()
        target = self.targets[idx]
        if self.is_train:
            flux = seq[0:6]
            err = seq[18:24]
            
            # Aug 1: Gaussian Noise Injection
            flux += torch.randn_like(flux) * 0.025 * torch.abs(flux)
            
            # Aug 2: Random Observation Dropout (Simulates bad weather)
            if random.random() < 0.3:
                drop_mask = torch.rand_like(flux) < 0.15
                flux[drop_mask] = 0.0
            
            # Aug 3: Global Scaling (Simulates distance differences)
            scale = random.uniform(0.85, 1.15)
            flux *= scale
            err *= scale
            
            # NEW Aug 4: Time Shifting (Prevents model from memorizing peak index)
            if random.random() < 0.5:
                shift = random.randint(-15, 15)
                flux = torch.roll(flux, shifts=shift, dims=1)
                err = torch.roll(err, shifts=shift, dims=1)
                if shift > 0: # Pad zeros to the space we opened up
                    flux[:, :shift] = 0; err[:, :shift] = 0
                else:
                    flux[:, shift:] = 0; err[:, shift:] = 0
            seq[0:6] = flux
            seq[18:24] = err
        return torch.nan_to_num(seq), torch.nan_to_num(stat), target

==========================================<br>
4. TRAINING: K-Fold Cross Validation<br>
==========================================

In [ ]:
if __name__ == "__main__":
    train_data_dict, scaler = preprocess_and_cache(
        os.path.join(CONFIG['DATA_DIR'], 'train_log.csv'), CONFIG['DATA_DIR'], "train", True
    )
    
    # Scale static features using standard Z-score normalization
    train_data_dict['statics'] = torch.tensor(scaler.transform(train_data_dict['statics'].numpy()), dtype=torch.float32)
    
    targets = train_data_dict['targets'].numpy()
    indices = np.arange(len(targets))
    
    # NEW: K-Fold Strategy initializes here
    skf = StratifiedKFold(n_splits=CONFIG['K_FOLDS'], shuffle=True, random_state=42)
    
    fold_models = []
    fold_f1_scores = []
    first_fold_history_train = []
    first_fold_history_val = []
    
    print(f"Starting {CONFIG['K_FOLDS']}-Fold TDE Hunt...")
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(indices, targets)):
        print(f"\n--- FOLD {fold+1}/{CONFIG['K_FOLDS']} ---")
        
        train_dataset = FastAstroDataset(train_data_dict, train_idx, True)
        val_dataset = FastAstroDataset(train_data_dict, val_idx, False)
        
        # Sampler to ensure 50/50 batches despite class imbalance
        train_targets = targets[train_idx]
        weights = 1. / np.array([len(np.where(train_targets == t)[0]) for t in np.unique(train_targets)])
        sampler = WeightedRandomSampler(torch.from_numpy(np.array([weights[t] for t in train_targets])).double(), len(train_targets))
        train_loader = DataLoader(train_dataset, batch_size=CONFIG['BATCH_SIZE'], sampler=sampler, num_workers=0)
        val_loader = DataLoader(val_dataset, batch_size=CONFIG['BATCH_SIZE'], num_workers=0)
        model = MultiModalCNN().to(device)
        
        # NEW: Differential Learning Rates (CNN learns faster than MLP, Classifier is middle ground)
        # Also increased weight_decay heavily to fight overfitting
        optimizer = optim.AdamW([
            {'params': model.cnn.parameters(), 'lr': 0.0006},
            {'params': model.mlp.parameters(), 'lr': 0.0001},
            {'params': model.classifier.parameters(), 'lr': 0.0003}
        ], weight_decay=1e-2)
        
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'max', patience=4, factor=0.5)
        criterion = FocalLoss()
        best_f1, best_th, best_model_wts = 0.0, 0.4, copy.deepcopy(model.state_dict())
        no_improve = 0
        for epoch in range(CONFIG['EPOCHS']):
            model.train()
            train_probs, train_truths = [], []
            
            for seq, stat, target in train_loader:
                optimizer.zero_grad()
                outputs = model(seq.to(device), stat.to(device))
                loss = criterion(outputs, target.to(device))
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0) # Prevents exploding gradients
                optimizer.step()
                
                p = torch.softmax(outputs, dim=1)[:, 1]
                train_probs.extend(p.detach().cpu().numpy())
                train_truths.extend(target.numpy())
                
            train_f1 = f1_score(train_truths, (np.array(train_probs) > 0.5).astype(int))
            if fold == 0: first_fold_history_train.append(train_f1)
            
            model.eval()
            probs, truths = [], []
            with torch.no_grad():
                for seq, stat, target in val_loader:
                    p = torch.softmax(model(seq.to(device), stat.to(device)), dim=1)[:, 1]
                    probs.extend(p.cpu().numpy())
                    truths.extend(target.numpy())
            
            # Sweeping for the best threshold (0.1 to 0.9)
            epoch_f1, epoch_th = 0.0, 0.3
            for th in np.arange(0.1, 0.9, 0.05):
                score = f1_score(truths, (np.array(probs) > th).astype(int))
                if score > epoch_f1:
                    epoch_f1, epoch_th = score, th
                    
            if fold == 0: first_fold_history_val.append(epoch_f1)
            scheduler.step(epoch_f1)
            
            if epoch_f1 > best_f1:
                best_f1, best_th, best_model_wts = epoch_f1, epoch_th, copy.deepcopy(model.state_dict())
                no_improve = 0
                print(f"F{fold+1} Best F1: {best_f1:.4f} @ TH {best_th:.2f} (Epoch {epoch+1})")
            else:
                no_improve += 1
            
            if no_improve >= CONFIG['EARLY_STOP_PATIENCE']:
                print(f"Early stopping triggered on Fold {fold+1}")
                break
        print(f"Fold {fold+1} Complete. Best F1: {best_f1:.4f}")
        model.load_state_dict(best_model_wts)
        fold_models.append((model, best_th))
        fold_f1_scores.append(best_f1)
    print(f"\nAll Folds Complete! Average Validation F1: {np.mean(fold_f1_scores):.4f}")

    # ==========================================
    # 5. GENERATE PRESENTATION GRAPHS
    # ==========================================
    print("Generating graphs for presentation slides...")
    sns.set_theme(style="whitegrid")
    
    # Graph A: Learning Curve (Using data from Fold 1)
    plt.figure(figsize=(10, 6))
    epochs_ran = range(1, len(first_fold_history_val) + 1)
    plt.plot(epochs_ran, first_fold_history_train, label='Training F1 (Fold 1)', marker='o', color='#1f77b4')
    plt.plot(epochs_ran, first_fold_history_val, label='Validation F1 (Fold 1)', marker='s', color='#ff7f0e')
    plt.title("Fold 1 Learning Curve", fontsize=16, fontweight='bold')
    plt.xlabel("Epoch", fontsize=12)
    plt.ylabel("F1 Score", fontsize=12)
    plt.legend(loc="lower right", fontsize=11)
    plt.tight_layout()
    plt.savefig("Learning_Curve_Slide.png", dpi=300)
    plt.close()
    
    # Graph B: K-Fold Consistency (Replaces Volume Knob graph)
    plt.figure(figsize=(8, 6))
    bars = plt.bar([f'Fold {i+1}' for i in range(CONFIG['K_FOLDS'])], fold_f1_scores, color='#2ca02c')
    for bar in bars:
        yval = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2, yval + 0.005, f'{yval:.3f}', ha='center', va='bottom', fontsize=12, fontweight='bold')
    plt.axhline(y=np.mean(fold_f1_scores), color='r', linestyle='--', label=f'Average: {np.mean(fold_f1_scores):.3f}')
    plt.title("Cross-Validation Stability Across 5 Folds", fontsize=16, fontweight='bold')
    plt.ylabel("Validation F1 Score", fontsize=12)
    plt.ylim(0.0, max(fold_f1_scores) + 0.05) 
    plt.legend()
    plt.tight_layout()
    plt.savefig("KFold_Performance_Slide.png", dpi=300)
    plt.close()

    # ==========================================
    # 6. TEST INFERENCE: Ensembling the Folds
    # ==========================================
    print("Running Ensembled Inference on test dataset...")
    test_data_dict, _ = preprocess_and_cache(
        os.path.join(CONFIG['DATA_DIR'], 'test_log.csv'), CONFIG['DATA_DIR'], "test", False
    )
    test_data_dict['statics'] = torch.tensor(scaler.transform(test_data_dict['statics'].numpy()), dtype=torch.float32)
    
    test_dataset = FastAstroDataset(test_data_dict, is_train=False)
    test_loader = DataLoader(test_dataset, batch_size=CONFIG['BATCH_SIZE'])
    
    final_preds = np.zeros(len(test_dataset))
    final_raw_probs = np.zeros(len(test_dataset)) # NEW: Array to hold raw probabilities
    
    # Get predictions and raw probabilities from all 5 models
    with torch.no_grad():
        for model, th in fold_models:
            model.eval()
            fold_preds = []
            fold_probs_raw = []
            for seq, stat, _ in test_loader:
                p = torch.softmax(model(seq.to(device), stat.to(device)), dim=1)[:, 1]
                
                # Store the binary vote (1 or 0) based on this fold's threshold
                fold_preds.extend((p.cpu().numpy() > th).astype(int))
                
                # Store the raw continuous probability
                fold_probs_raw.extend(p.cpu().numpy())
                
            final_preds += np.array(fold_preds)
            final_raw_probs += np.array(fold_probs_raw)
            
    # Majority Voting: If 3 or more folds say it's a TDE, it's a TDE.
    ensemble_decisions = (final_preds >= (CONFIG['K_FOLDS'] / 2)).astype(int)
    
    # Average the raw probabilities across all folds
    average_probs = final_raw_probs / CONFIG['K_FOLDS']

    # Save the binary decisions
    submission_path = 'submission.csv'
    pd.DataFrame({'object_id': test_dataset.ids, 'prediction': ensemble_decisions}).to_csv(submission_path, index=False)
    
    # Save the continuous probabilities
    prob_path = 'cnn_probability.csv'
    pd.DataFrame({'object_id': test_dataset.ids, 'probability': average_probs}).to_csv(prob_path, index=False)
    
    print(f"Submission saved! Predicted {sum(ensemble_decisions)} TDEs via Ensembling.")
    print(f"Probabilities saved to {prob_path}!")